# LeetCode 932 — Beautiful Array bằng Python

Link đề: https://leetcode.com/problems/beautiful-array/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Math (toán), Divide and Conquer (chia để trị)

Skeleton LeetCode:

```python
class Solution:
    def beautifulArray(self, n: int) -> List[int]:
        ...
```

Ghi chú: mảng **đẹp** là hoán vị của `1..n` sao cho **không có ba chỉ số `i < k < j`** với `2 * nums[k] == nums[i] + nums[j]` (không có phần tử nào nằm giữa và là trung bình cộng của hai phần tử hai bên). Đề yêu cầu trả về **một** mảng đẹp bất kỳ; vì có **nhiều đáp án đúng**, ta kiểm thử bằng **hàm kiểm tra tính hợp lệ** chứ không so với đáp án mẫu. Ý tưởng chìa khóa: `nums[i] + nums[j]` **phải chẵn** thì mới có thể bằng `2 * nums[k]`, nên **đặt mọi số lẻ bên trái và mọi số chẵn bên phải** thì không có cặp (lẻ, chẵn) nào bị phạm; còn trong từng nửa ta dựng đệ quy từ một mảng đẹp nhỏ hơn bằng phép biến đổi tuyến tính.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: dựng một hoán vị của `1..n` thỏa điều kiện "không có phần tử nằm giữa bằng trung bình cộng của hai phần tử hai bên".
2. Nắm các công cụ Python cần thiết: hoán vị `itertools.permutations`, ánh xạ vị trí `pos[v]`, list comprehension để biến đổi `x -> 2x - 1`, `x -> 2x`, chia để trị trên kích thước `(n + 1) // 2` và `n // 2`, và dịch bit/đảo bit.
3. Hiểu nguyên lý: **phép biến đổi tuyến tính `x -> a*x + b` (`a != 0`) và phép đảo mảng giữ nguyên tính đẹp**; số lẻ và số chẵn **không bao giờ vi phạm lẫn nhau** vì tổng của một lẻ và một chẵn là lẻ.
4. Đi qua 4 cách: quay lui dựng mảng đẹp (brute), chia để trị đệ quy (cách nộp chính), nhân đôi lặp từ dưới lên, và **đảo bit** của chỉ số; kiểm thử bằng hai hàm kiểm tra độc lập.
5. Tránh các lỗi kinh điển: dùng mảng tăng dần/lẻ rồi chẵn không biến đổi từ mảng nhỏ, chia sai kích thước hai nửa, quên lọc giá trị `> n` khi lặp, và kiểm thử bằng so sánh với đáp án mẫu.

## 2. Đề bài & ví dụ

Mảng `nums` độ dài `n` là **đẹp** nếu:

- `nums` là một **hoán vị** của các số nguyên trong `[1, n]`;
- với mọi `0 <= i < j < n`, **không tồn tại** chỉ số `k` với `i < k < j` sao cho `2 * nums[k] == nums[i] + nums[j]`.

Cho số nguyên `n`, trả về **một** mảng đẹp bất kỳ độ dài `n`. Đảm bảo luôn có ít nhất một đáp án.

**Ví dụ 1**

- Input: `n = 4`
- Output: `[2,1,4,3]` (còn nhiều đáp án khác, ví dụ `[1,3,2,4]`)

**Ví dụ 2**

- Input: `n = 5`
- Output: `[3,1,2,5,4]`

**Ràng buộc:**

- `1 <= n <= 1000`

In [1]:
from typing import List
from itertools import permutations
import random, time

def is_beautiful(arr: List[int], n: int) -> bool:
    """Kiểm tra hợp lệ O(n^2): hoán vị của 1..n và không có k nằm giữa i, j với 2*arr[k] == arr[i] + arr[j]."""
    if len(arr) != n or sorted(arr) != list(range(1, n + 1)):
        return False
    pos = {v: i for i, v in enumerate(arr)}
    for i in range(n):
        for j in range(i + 2, n):
            s = arr[i] + arr[j]
            if s % 2 == 0:
                k = pos[s // 2]
                if i < k < j:
                    return False
    return True

def is_beautiful_cubic(arr: List[int], n: int) -> bool:
    """Kiểm tra độc lập O(n^3), theo đúng định nghĩa (ba vòng lặp)."""
    if len(arr) != n or sorted(arr) != list(range(1, n + 1)):
        return False
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(i + 1, j):
                if 2 * arr[k] == arr[i] + arr[j]:
                    return False
    return True

examples = [4, 5]
for n, sample in ((4, [2, 1, 4, 3]), (5, [3, 1, 2, 5, 4])):
    print(f"n={n}: đáp án mẫu {sample} hợp lệ? {is_beautiful(sample, n)} {is_beautiful_cubic(sample, n)}")
    assert is_beautiful(sample, n) and is_beautiful_cubic(sample, n)
print("[1, 2, 3] hợp lệ?", is_beautiful([1, 2, 3], 3), "(2 * 2 = 1 + 3)")
assert is_beautiful([1, 2, 3], 3) is False

n=4: đáp án mẫu [2, 1, 4, 3] hợp lệ? True True
n=5: đáp án mẫu [3, 1, 2, 5, 4] hợp lệ? True True
[1, 2, 3] hợp lệ? False (2 * 2 = 1 + 3)


## 3. Kiến thức nền

### 3.1. Điều kiện cần cho `2 * nums[k] == nums[i] + nums[j]`

Vế trái `2 * nums[k]` luôn **chẵn**, nên `nums[i] + nums[j]` phải chẵn: `nums[i]` và `nums[j]` **cùng tính chẵn lẻ**. Nếu một số lẻ và một số chẵn thì tổng lẻ, không thể bằng `2 * nums[k]`: cặp đó **an toàn**.

In [2]:
for a, b in ((1, 4), (3, 6), (2, 5), (1, 3), (2, 4)):
    print(f"{a} + {b} = {a + b}: {'chẵn, có thể vi phạm' if (a + b) % 2 == 0 else 'lẻ, luôn an toàn'}")

1 + 4 = 5: lẻ, luôn an toàn
3 + 6 = 9: lẻ, luôn an toàn
2 + 5 = 7: lẻ, luôn an toàn
1 + 3 = 4: chẵn, có thể vi phạm
2 + 4 = 6: chẵn, có thể vi phạm


### 3.2. Biến đổi tuyến tính giữ tính đẹp

Nếu `A` đẹp thì `B[i] = a * A[i] + b` (`a != 0`) cũng thỏa điều kiện **không có trung bình cộng nằm giữa**: `2 * B[k] == B[i] + B[j]` ⇔ `2 * (a A[k] + b) == a A[i] + a A[j] + 2b` ⇔ `2 A[k] == A[i] + A[j]`. Vậy `x -> 2x - 1` biến một mảng đẹp thành mảng các số **lẻ** đẹp, `x -> 2x` thành mảng các số **chẵn** đẹp (nhưng chưa nhất thiết là hoán vị của `1..n`: ta kiểm soát điều đó bằng kích thước).

In [3]:
A = [1, 3, 2, 4]
print("A =", A, is_beautiful(A, 4))
odd = [2 * x - 1 for x in A]
even = [2 * x for x in A]
print("lẻ  =", odd, "chẵn =", even)
print("đảo mảng:", A[::-1], is_beautiful(A[::-1], 4))

A = [1, 3, 2, 4] True
lẻ  = [1, 5, 3, 7] chẵn = [2, 6, 4, 8]
đảo mảng: [4, 2, 3, 1] True


### 3.3. Ghép phần lẻ và phần chẵn

Nếu `L` (các số lẻ) và `R` (các số chẵn) đều đẹp (không có trung bình cộng nằm giữa trong từng phần) thì `L + R` đẹp: một cặp `(i, j)` nằm cùng trong `L` hay cùng trong `R` thì không vi phạm; một cặp lệch (`i` trong `L`, `j` trong `R`) có tổng lẻ nên không bao giờ bằng `2 * nums[k]`.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Dựng mảng đẹp cho `n = 4` từ `n = 2`:

- Mảng đẹp cho `n = 1`: `[1]`; cho `n = 2`: `[1, 2]` (không có `k` nằm giữa).
- `n = 4`: cần các số lẻ `{1, 3}` và chẵn `{2, 4}`: lấy mảng đẹp `[1, 2]`, biến thành lẻ `[2*1 - 1, 2*2 - 1] = [1, 3]` và chẵn `[2*1, 2*2] = [2, 4]`; ghép: `[1, 3, 2, 4]`.
- Kiểm tra: các cặp cùng chẵn lẻ: `(1, 3)` nằm cạnh nhau (không có `k` giữa), `(2, 4)` cạnh nhau: không vi phạm. Cặp lệch có tổng lẻ.

Với `n = 5`: số lẻ `{1, 3, 5}` (3 số) và chẵn `{2, 4}` (2 số): ta cần mảng đẹp cho `3` (lẻ) và cho `2` (chẵn), tức `(n + 1) // 2 = 3` và `n // 2 = 2`.

## 5. Cách 1 — Brute Force: quay lui dựng mảng đẹp

**Ý tưởng.** Dựng hoán vị từ trái sang phải: thử mọi số chưa dùng; khi thêm số `v` ở vị trí `j`, kiểm tra không tạo ra `i < k < j` với `2 * arr[k] == arr[i] + v`. Nếu bế tắc thì quay lui. Hoạt động đúng nhưng **mũ**: chỉ dùng được với `n` nhỏ; vì vậy cũng là **oracle** cho `n` nhỏ (kiểm tra bằng định nghĩa).

In [4]:
def beautiful_backtrack(n: int) -> List[int]:
    arr: List[int] = []
    used = [False] * (n + 1)

    def ok(v: int) -> bool:
        j = len(arr)
        for i in range(j):
            s = arr[i] + v
            if s % 2 == 0:
                mid = s // 2
                # mid nằm giữa i và j?
                for k in range(i + 1, j):
                    if arr[k] == mid:
                        return False
        return True

    def dfs() -> bool:
        if len(arr) == n:
            return True
        for v in range(1, n + 1):
            if not used[v] and ok(v):
                used[v] = True
                arr.append(v)
                if dfs():
                    return True
                arr.pop()
                used[v] = False
        return False

    assert dfs()
    return arr

for n in examples:
    got = beautiful_backtrack(n)
    print(n, "->", got, "hợp lệ" if is_beautiful(got, n) else "SAI")
    assert is_beautiful(got, n) and is_beautiful_cubic(got, n)

4 -> [1, 3, 2, 4] hợp lệ
5 -> [1, 5, 3, 2, 4] hợp lệ


**Độ phức tạp và nút thắt.** Số nhánh thử tăng rất nhanh. Xem thời gian tìm **một** mảng đẹp và số mảng đẹp theo `n` (đếm bằng liệt kê hoán vị):

In [5]:
def count_beautiful_bruteforce(n: int) -> int:
    return sum(1 for p in permutations(range(1, n + 1)) if is_beautiful(list(p), n))

print("số mảng đẹp n = 1..8:", [count_beautiful_bruteforce(n) for n in range(1, 9)])
for n in (8, 10, 12, 13):
    t0 = time.perf_counter()
    beautiful_backtrack(n)
    print(f"quay lui tìm một mảng đẹp, n={n:2}: {1000 * (time.perf_counter() - t0):9.1f} ms")

số mảng đẹp n = 1..8: [1, 2, 4, 10, 20, 48, 104, 282]
quay lui tìm một mảng đẹp, n= 8:       0.4 ms
quay lui tìm một mảng đẹp, n=10:      20.0 ms


quay lui tìm một mảng đẹp, n=12:     281.3 ms


quay lui tìm một mảng đẹp, n=13:     874.1 ms


Với `n = 1000` quay lui không khả thi. Nút thắt: ta đang **tìm** mảng đẹp trong khi có thể **xây** nó có cấu trúc.

## 6. Ý tưởng thuật toán / bước đột phá

**Chia để trị theo tính chẵn lẻ.**

- Mảng đẹp cho `n` = (các số lẻ của `1..n` sắp xếp theo một mảng đẹp) + (các số chẵn của `1..n` sắp xếp theo một mảng đẹp).
- Có `(n + 1) // 2` số lẻ `1, 3, ..., ` và `n // 2` số chẵn `2, 4, ...`.
- Số lẻ thứ `t` là `2t - 1`, số chẵn thứ `t` là `2t`. Nếu `A` là mảng đẹp của `(n + 1) // 2` thì `[2x - 1 for x in A]` là mảng lẻ đẹp (chứa đúng các số lẻ); nếu `B` là mảng đẹp của `n // 2` thì `[2x for x in B]` là mảng chẵn đẹp.

Từ đó `F(n) = [2x - 1 for x in F((n + 1) // 2)] + [2x for x in F(n // 2)]` với `F(1) = [1]`. Có `log2 n` tầng; mỗi tầng tốn `O(n)` ⇒ `O(n)` tổng cộng (chuỗi hình học).

| Cách | Ý tưởng |
|---|---|
| 2 | Đệ quy `F(n)` theo công thức trên (nộp) |
| 3 | Lặp từ dưới lên: nhân đôi mảng rồi lọc giá trị `<= n` |
| 4 | Đảo bit của chỉ số trong một lũy thừa của 2 rồi lọc `<= n` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Chia để trị đệ quy (cách nộp chính)

**Giải thích từng đoạn.**

- `n == 1` trả `[1]`.
- `left = F((n + 1) // 2)`, `right = F(n // 2)`.
- Biến đổi: `left -> [2x - 1]`, `right -> [2x]`, rồi ghép `left + right`.
- Chiều sâu đệ quy `O(log n)`.

In [6]:
def beautiful_recursive(n: int) -> List[int]:
    if n == 1:
        return [1]
    left = beautiful_recursive((n + 1) // 2)
    right = beautiful_recursive(n // 2)
    return [2 * x - 1 for x in left] + [2 * x for x in right]

for n in examples:
    got = beautiful_recursive(n)
    print(n, "->", got, "hợp lệ" if is_beautiful(got, n) else "SAI")
    assert is_beautiful(got, n)

4 -> [1, 3, 2, 4] hợp lệ
5 -> [1, 5, 3, 2, 4] hợp lệ


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def beautifulArray(self, n: int) -> List[int]:
        if n == 1:
            return [1]
        left = self.beautifulArray((n + 1) // 2)
        right = self.beautifulArray(n // 2)
        return [2 * x - 1 for x in left] + [2 * x for x in right]

sol_rec = Solution().beautifulArray
assert all(is_beautiful(sol_rec(n), n) for n in examples)
print("Cách 2 qua ví dụ (kiểm tra bằng hàm hợp lệ)")

Cách 2 qua ví dụ (kiểm tra bằng hàm hợp lệ)


**Độ phức tạp (Cách 2):** thời gian `O(n)` (tổng kích thước các mảng ở các tầng là `n + n/2 + n/4 + ... = O(n)`), bộ nhớ `O(n)`.

---

### 7.2. Cách 3 — Lặp từ dưới lên: nhân đôi và lọc

**Ý tưởng khác.** Bắt đầu `arr = [1]`. Lặp tới khi `len(arr) == n`: `arr = [2x - 1 for x in arr if 2x - 1 <= n] + [2x for x in arr if 2x <= n]`. Mỗi vòng độ dài gần như nhân đôi (và bị chặn bởi `n`). Không dùng đệ quy.

**Giải thích từng đoạn.** Điều kiện lọc `<= n` giữ cho mảng chỉ chứa giá trị hợp lệ; ở vòng cuối mảng đủ `n` phần tử gồm đúng `1..n`.

In [8]:
def beautiful_iterative(n: int) -> List[int]:
    arr = [1]
    while len(arr) < n:
        arr = [2 * x - 1 for x in arr if 2 * x - 1 <= n] + [2 * x for x in arr if 2 * x <= n]
    return arr

for n in examples:
    got = beautiful_iterative(n)
    print(n, "->", got, "hợp lệ" if is_beautiful(got, n) else "SAI")
    assert is_beautiful(got, n)

4 -> [1, 3, 2, 4] hợp lệ
5 -> [1, 5, 3, 2, 4] hợp lệ


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def beautifulArray(self, n: int) -> List[int]:
        arr = [1]
        while len(arr) < n:
            arr = [2 * x - 1 for x in arr if 2 * x - 1 <= n] + [2 * x for x in arr if 2 * x <= n]
        return arr

sol_iter = Solution().beautifulArray
assert all(is_beautiful(sol_iter(n), n) for n in examples)
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n)` thời gian (tổng các vòng là chuỗi hình học), `O(n)` bộ nhớ.

---

### 7.3. Cách 4 — Đảo bit của chỉ số

**Ý tưởng khác.** Khi `n = 2^b`, vòng lặp nhân đôi cho `arr[i] = reverse_bits(i, b) + 1` (đảo thứ tự `b` bit của chỉ số `i`). Với `n` bất kỳ, lấy `P = 2^b >= n`, liệt kê `i = 0..P-1`, tính `v = reverse_bits(i, b) + 1` và **giữ** những `v <= n` theo thứ tự `i`. Kết quả cũng đẹp (là mảng đẹp của `P` rồi **xóa** các phần tử `> n`: xóa phần tử không làm xuất hiện ba chỉ số vi phạm mới vì mọi bộ ba còn lại là bộ ba của mảng gốc).

**Giải thích từng đoạn.** `reverse_bits(i, b)`: đảo `b` bit thấp của `i`. Mảng của `P` là hoán vị của `1..P` đẹp; lọc `v <= n` giữ đúng các giá trị `1..n`.

In [10]:
def reverse_bits(i: int, bits: int) -> int:
    r = 0
    for _ in range(bits):
        r = (r << 1) | (i & 1)
        i >>= 1
    return r

def beautiful_bitreverse(n: int) -> List[int]:
    bits = max(1, (n - 1).bit_length())
    P = 1 << bits
    return [v for i in range(P) for v in (reverse_bits(i, bits) + 1,) if v <= n]

for n in examples:
    got = beautiful_bitreverse(n)
    print(n, "->", got, "hợp lệ" if is_beautiful(got, n) else "SAI")
    assert is_beautiful(got, n)
print("n = 8:", beautiful_bitreverse(8))

4 -> [1, 3, 2, 4] hợp lệ
5 -> [1, 5, 3, 2, 4] hợp lệ
n = 8: [1, 5, 3, 7, 2, 6, 4, 8]


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def beautifulArray(self, n: int) -> List[int]:
        bits = max(1, (n - 1).bit_length())
        ans = []
        for i in range(1 << bits):
            r, x = 0, i
            for _ in range(bits):
                r = (r << 1) | (x & 1)
                x >>= 1
            if r + 1 <= n:
                ans.append(r + 1)
        return ans

sol_bits = Solution().beautifulArray
assert all(is_beautiful(sol_bits(n), n) for n in examples)
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(P log P)` với `P < 2n` (mỗi chỉ số đảo `log P` bit), `O(n)` bộ nhớ.

## 8. TRACE từng bước

### 8.1. Cách 2 cho `n = 5`

In [12]:
def trace_recursive(n, depth=0):
    pad = "  " * depth
    if n == 1:
        print(f"{pad}F(1) = [1]")
        return [1]
    print(f"{pad}F({n}): trái = F({(n + 1) // 2}), phải = F({n // 2})")
    left = trace_recursive((n + 1) // 2, depth + 1)
    right = trace_recursive(n // 2, depth + 1)
    res = [2 * x - 1 for x in left] + [2 * x for x in right]
    print(f"{pad}F({n}): lẻ {[2 * x - 1 for x in left]} + chẵn {[2 * x for x in right]} = {res}")
    return res

res = trace_recursive(5)
assert is_beautiful(res, 5) and res == [1, 5, 3, 2, 4]

F(5): trái = F(3), phải = F(2)
  F(3): trái = F(2), phải = F(1)
    F(2): trái = F(1), phải = F(1)
      F(1) = [1]
      F(1) = [1]
    F(2): lẻ [1] + chẵn [2] = [1, 2]
    F(1) = [1]
  F(3): lẻ [1, 3] + chẵn [2] = [1, 3, 2]
  F(2): trái = F(1), phải = F(1)
    F(1) = [1]
    F(1) = [1]
  F(2): lẻ [1] + chẵn [2] = [1, 2]
F(5): lẻ [1, 5, 3] + chẵn [2, 4] = [1, 5, 3, 2, 4]


### 8.2. Cách 3 (nhân đôi và lọc) cho `n = 5`

`[1]` -> `[1, 2]` -> vòng kế: lẻ `[1, 3]`, chẵn `[2, 4]` ghép `[1, 3, 2, 4]` (độ dài 4 < 5) -> vòng kế: lẻ `[1, 5, 3]` (lọc `<= 5`: `2*1-1 = 1`, `2*3-1 = 5`, `2*2-1 = 3`, `2*4-1 = 7` bị loại), chẵn `[2, 6, 4, 8]` lọc `[2, 4]`: ghép `[1, 5, 3, 2, 4]`.

In [13]:
def trace_iterative(n):
    arr = [1]
    print("bắt đầu", arr)
    while len(arr) < n:
        odd = [2 * x - 1 for x in arr if 2 * x - 1 <= n]
        even = [2 * x for x in arr if 2 * x <= n]
        arr = odd + even
        print(f"lẻ {odd} + chẵn {even} -> {arr}")
    return arr

assert trace_iterative(5) == [1, 5, 3, 2, 4]

bắt đầu [1]
lẻ [1] + chẵn [2] -> [1, 2]
lẻ [1, 3] + chẵn [2, 4] -> [1, 3, 2, 4]
lẻ [1, 5, 3] + chẵn [2, 4] -> [1, 5, 3, 2, 4]


### 8.3. Cách 4 (đảo bit) cho `n = 5`

`bits = 3`, `P = 8`: các `v = reverse_bits(i, 3) + 1` cho `i = 0..7`: `1, 5, 3, 7, 2, 6, 4, 8`; lọc `<= 5`: `1, 5, 3, 2, 4`: trùng Cách 2 và 3.

In [14]:
bits = 3
full = [reverse_bits(i, bits) + 1 for i in range(1 << bits)]
print("mảng của P = 8:", full, "-> lọc <= 5:", [v for v in full if v <= 5])
assert [v for v in full if v <= 5] == beautiful_bitreverse(5) == [1, 5, 3, 2, 4]

mảng của P = 8: [1, 5, 3, 7, 2, 6, 4, 8] -> lọc <= 5: [1, 5, 3, 2, 4]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (chia để trị đệ quy): ngắn nhất, đúng công thức `F(n) = [2x - 1 | x in F((n + 1)/2)] + [2x | x in F(n/2)]`.

```python
class Solution:
    def beautifulArray(self, n: int) -> List[int]:
        if n == 1:
            return [1]
        left = self.beautifulArray((n + 1) // 2)
        right = self.beautifulArray(n // 2)
        return [2 * x - 1 for x in left] + [2 * x for x in right]
```

In [15]:
class Solution:
    def beautifulArray(self, n: int) -> List[int]:
        if n == 1:
            return [1]
        left = self.beautifulArray((n + 1) // 2)
        right = self.beautifulArray(n // 2)
        return [2 * x - 1 for x in left] + [2 * x for x in right]

sol = Solution()
for n in examples:
    got = sol.beautifulArray(n)
    print(n, "->", got, "hợp lệ" if is_beautiful(got, n) else "SAI")
    assert is_beautiful(got, n) and is_beautiful_cubic(got, n)

4 -> [1, 3, 2, 4] hợp lệ
5 -> [1, 5, 3, 2, 4] hợp lệ


## 10. Vì sao đúng

**Bổ đề 1 (biến đổi tuyến tính).** Nếu `A` là dãy không có chỉ số `i < k < j` với `2A[k] = A[i] + A[j]` thì với `a != 0`, `b` tùy ý, `B[t] = a A[t] + b` cũng không có: `2B[k] = B[i] + B[j]` ⇔ `2aA[k] + 2b = aA[i] + aA[j] + 2b` ⇔ `2A[k] = A[i] + A[j]`.

**Bổ đề 2 (ghép lẻ + chẵn).** Cho `L` gồm toàn số lẻ, `R` gồm toàn số chẵn, mỗi phần "không có trung bình cộng nằm giữa". Xét bộ ba `i < k < j` của `L + R`:
   - nếu `i, j` cùng thuộc `L` (hoặc cùng thuộc `R`) thì `k` nằm giữa chúng cũng thuộc phần đó, và theo giả thiết `2 nums[k] != nums[i] + nums[j]`;
   - nếu `i` thuộc `L`, `j` thuộc `R` thì `nums[i] + nums[j]` là lẻ, còn `2 nums[k]` chẵn nên không thể bằng.
   Vậy `L + R` không có bộ ba vi phạm.

**Định lý (tính đúng của `F(n)`).** Quy nạp theo `n`. `F(1) = [1]` đẹp. Giả sử `F(m)` đẹp (hoán vị của `1..m`) với mọi `m < n`, `n > 1`. Đặt `p = (n + 1) // 2 < n`, `q = n // 2 < n` (với `n >= 2`). Theo giả thiết quy nạp `F(p)` là hoán vị đẹp của `1..p`; `[2x - 1 for x in F(p)]` là hoán vị của các số lẻ `1, 3, ..., 2p - 1` (đúng là mọi số lẻ trong `1..n` vì `2p - 1 = n` nếu `n` lẻ, `n - 1` nếu `n` chẵn), và đẹp theo Bổ đề 1. Tương tự `[2x for x in F(q)]` là hoán vị của `2, 4, ..., 2q` (mọi số chẵn trong `1..n`), đẹp theo Bổ đề 1. Ghép theo Bổ đề 2 cho hoán vị đẹp của `1..n`. Hai phần cùng chứa `p + q = n` số khác nhau.

**Cách 3 (lặp).** Mỗi vòng áp dụng đúng phép dựng trên với kích thước `p`, `q` nhưng nhận được bằng **lọc** giá trị `<= n` của mảng đẹp lớn hơn; lọc một mảng đẹp (xóa phần tử) không tạo bộ ba vi phạm mới (mọi bộ ba còn lại là bộ ba của mảng gốc), nên vẫn đẹp.

**Cách 4 (đảo bit).** Với `n = 2^b` công thức nhân đôi cho `arr[i] = reverse_bits(i, b) + 1` (quy nạp theo `b`: mảng cho `2^b` là `[2x - 1 for x in A] + [2x for x in A]` với `A` ứng với `2^(b-1)`; nửa trái ứng với chỉ số có bit cao nhất `0`, nửa phải bit cao nhất `1`; ánh xạ `x -> 2x - 1`/`2x` thêm một bit thấp vào giá trị đảo). Với `n` bất kỳ lọc `<= n` như trên.

**Kiểm chứng bằng máy.** Hai hàm kiểm tra **độc lập nhau** theo đúng định nghĩa: `is_beautiful` (`O(n^2)`, dùng bảng vị trí) và `is_beautiful_cubic` (`O(n^3)`, ba vòng lặp thuần định nghĩa) cho `n` nhỏ; và với mọi `n` từ `1` đến `1000` mọi lời giải trả về mảng thỏa `is_beautiful`. Vì có nhiều đáp án đúng, **không** so với đáp án mẫu.

In [16]:
rng = random.Random(1932)
# hai hàm kiểm tra khớp nhau trên hoán vị ngẫu nhiên và trên các mảng đẹp
for _ in range(400):
    n = rng.randint(1, 9)
    p = list(range(1, n + 1))
    rng.shuffle(p)
    assert is_beautiful(p, n) == is_beautiful_cubic(p, n), p
for n in range(1, 8):
    for p in permutations(range(1, n + 1)):
        assert is_beautiful(list(p), n) == is_beautiful_cubic(list(p), n)
print("hai hàm kiểm tra (O(n^2) và O(n^3)) khớp nhau trên mọi hoán vị với n <= 7 và 400 hoán vị ngẫu nhiên n <= 9")

hai hàm kiểm tra (O(n^2) và O(n^3)) khớp nhau trên mọi hoán vị với n <= 7 và 400 hoán vị ngẫu nhiên n <= 9


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Quay lui | mũ (không dùng được với `n` lớn) | `O(n)` |
| 2. Đệ quy chia để trị | `O(n)` | `O(n)` |
| 3. Lặp nhân đôi và lọc | `O(n)` | `O(n)` |
| 4. Đảo bit | `O(n log n)` | `O(n)` |

Đo thử với `n = 1000` (cỡ tối đa) và với `n` lớn hơn để thấy xu hướng:

In [17]:
for n in (1000, 10 ** 4, 10 ** 5):
    row = []
    for name, fn in (("Cách 2", sol_rec), ("Cách 3", sol_iter), ("Cách 4", sol_bits)):
        t0 = time.perf_counter()
        arr = fn(n)
        row.append(f"{name}: {1000 * (time.perf_counter() - t0):8.2f} ms")
        assert sorted(arr) == list(range(1, n + 1))
    print(f"n={n:6}: " + " | ".join(row))
arr = sol_rec(1000)
t0 = time.perf_counter()
print("kiểm tra n = 1000 bằng is_beautiful (O(n^2)):", is_beautiful(arr, 1000), f"{1000 * (time.perf_counter() - t0):.1f} ms")

n=  1000: Cách 2:     0.93 ms | Cách 3:     0.22 ms | Cách 4:     1.42 ms
n= 10000: Cách 2:    11.91 ms | Cách 3:     4.12 ms | Cách 4:    23.96 ms


n=100000: Cách 2:   130.30 ms | Cách 3:    29.96 ms | Cách 4:   273.77 ms
kiểm tra n = 1000 bằng is_beautiful (O(n^2)): True 73.4 ms


## 12. Bộ test

Vì **nhiều đáp án đều đúng**, mọi kiểm tra dùng **hàm kiểm tra tính hợp lệ** (`is_beautiful`, và `is_beautiful_cubic` cho `n` nhỏ), **không** so với đáp án mẫu. Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, kiểm tra mọi `n` từ `1` đến `1000`, và tính chất.

In [18]:
solutions = {
    "recursive": beautiful_recursive,
    "iterative": beautiful_iterative,
    "bitreverse": beautiful_bitreverse,
    "sol_rec": sol_rec,
    "sol_iter": sol_iter,
    "sol_bits": sol_bits,
    "final": Solution().beautifulArray,
}

def run_all(n, cubic=True):
    for name, fn in solutions.items():
        arr = fn(n)
        assert is_beautiful(arr, n), (name, n, arr[:20])
        if cubic and n <= 60:
            assert is_beautiful_cubic(arr, n), (name, n)
    return True

# 12.1 ví dụ đề: đáp án mẫu của đề cũng hợp lệ (có nhiều đáp án đúng)
for n in examples:
    run_all(n)
assert is_beautiful([2, 1, 4, 3], 4) and is_beautiful([3, 1, 2, 5, 4], 5)
print("12.1 ví dụ đề: OK; lời giải trả về", Solution().beautifulArray(4), "và", Solution().beautifulArray(5), "khác đáp án mẫu nhưng hợp lệ")

# 12.2 edge case
for n in (1, 2, 3, 4, 7, 8, 9, 15, 16, 17, 31, 32, 33, 63, 64, 65, 127, 128, 129, 255, 256, 257):
    run_all(n)
assert Solution().beautifulArray(1) == [1]
assert sorted(Solution().beautifulArray(2)) == [1, 2]
print("12.2 edge case (n = 1, các lũy thừa 2 và lân cận tới 257): OK")

12.1 ví dụ đề: OK; lời giải trả về [1, 3, 2, 4] và [1, 5, 3, 2, 4] khác đáp án mẫu nhưng hợp lệ
12.2 edge case (n = 1, các lũy thừa 2 và lân cận tới 257): OK


In [19]:
# 12.3 phản ví dụ: các cách sai (xác nhận hàm kiểm tra bắt được lỗi, lời giải đúng không bị lỗi)
def wrong_identity(n):                              # mảng tăng dần
    return list(range(1, n + 1))

def wrong_sorted_odd_even(n):                       # lẻ tăng dần rồi chẵn tăng dần (không biến đổi từ mảng nhỏ)
    return list(range(1, n + 1, 2)) + list(range(2, n + 1, 2))

def wrong_split_sizes(n):                           # chia sai kích thước: n // 2 cho cả hai nửa
    if n == 1:
        return [1]
    left = wrong_split_sizes(n // 2)
    right = wrong_split_sizes(n // 2)
    return [2 * x - 1 for x in left] + [2 * x for x in right]

def wrong_no_filter(n):                             # lặp nhân đôi nhưng quên lọc giá trị > n
    arr = [1]
    while len(arr) < n:
        arr = [2 * x - 1 for x in arr] + [2 * x for x in arr]
    return arr[:n]

def wrong_reverse_halves(n):                        # chẵn trước nhưng không biến đổi: chẵn tăng dần + lẻ tăng dần
    return list(range(2, n + 1, 2)) + list(range(1, n + 1, 2))

def first_bad(fn, upto=40):
    for n in range(1, upto + 1):
        try:
            arr = fn(n)
        except Exception:
            return n
        if not is_beautiful(arr, n):
            return n
    return None

counters = {}
for name, fn in (("mảng tăng dần", wrong_identity), ("lẻ rồi chẵn tăng dần (không biến đổi)", wrong_sorted_odd_even),
                 ("chia sai kích thước", wrong_split_sizes), ("quên lọc giá trị > n", wrong_no_filter),
                 ("chẵn rồi lẻ tăng dần", wrong_reverse_halves)):
    n = first_bad(fn)
    assert n is not None, name
    counters[name] = n
    arr = fn(n)
    print(f"{name:38} n={n:2}: {arr[:12]} hợp lệ? {is_beautiful(arr, n)}")
    assert run_all(n)
print("12.3 phản ví dụ: OK")

mảng tăng dần                          n= 3: [1, 2, 3] hợp lệ? False
lẻ rồi chẵn tăng dần (không biến đổi)  n= 5: [1, 3, 5, 2, 4] hợp lệ? False
chia sai kích thước                    n= 3: [1, 2] hợp lệ? False
quên lọc giá trị > n                   n= 5: [1, 5, 3, 7, 2] hợp lệ? False
chẵn rồi lẻ tăng dần                   n= 5: [2, 4, 1, 3, 5] hợp lệ? False
12.3 phản ví dụ: OK


In [20]:
# 12.4 mọi n từ 1 đến 80 (kiểm tra O(n^2), và O(n^3) cho n <= 40) cho cả 7 lời giải; thêm mẫu n lớn tới 1000
t0 = time.perf_counter()
for n in range(1, 81):
    for name, fn in solutions.items():
        arr = fn(n)
        assert is_beautiful(arr, n), (name, n)
        if n <= 40:
            assert is_beautiful_cubic(arr, n), (name, n)
rng4 = random.Random(4932)
big_ns = sorted(set(rng4.sample(range(81, 1001), 25) + [255, 256, 257, 511, 512, 513, 999, 1000]))
for n in big_ns:
    for name in ("recursive", "iterative", "bitreverse", "final"):
        assert is_beautiful(solutions[name](n), n), (name, n)
print(f"12.4 n = 1..80 (7 lời giải) và {len(big_ns)} giá trị n lớn tới 1000: OK ({time.perf_counter() - t0:.1f} s)")

# 12.5 các lời giải đệ quy/lặp/đảo bit cho cùng kết quả hay chỉ cùng hợp lệ?
same = [n for n in range(1, 200) if beautiful_recursive(n) == beautiful_iterative(n) == beautiful_bitreverse(n)]
print("số n < 200 mà ba cách cho CÙNG một mảng:", len(same), "/ 199")
assert len(same) == 199

# 12.6 tính chất: biến đổi tuyến tính / đảo mảng giữ tính đẹp; xóa phần tử giữ tính không có trung bình giữa
rng = random.Random(6932)
def no_avg_between(arr):
    pos = {v: i for i, v in enumerate(arr)}
    for i in range(len(arr)):
        for j in range(i + 2, len(arr)):
            s = arr[i] + arr[j]
            if s % 2 == 0 and s // 2 in pos and i < pos[s // 2] < j:
                return False
    return True

for n in (5, 17, 50, 128, 300):
    A = Solution().beautifulArray(n)
    assert no_avg_between(A[::-1])
    assert no_avg_between([3 * x + 7 for x in A])
    assert no_avg_between([-x for x in A])
    B = A[:]
    for _ in range(n // 3):
        B.pop(rng.randrange(len(B)))
    assert no_avg_between(B)
print("12.6 tính chất (đảo, biến đổi tuyến tính, xóa phần tử): OK")

12.4 n = 1..80 (7 lời giải) và 32 giá trị n lớn tới 1000: OK (2.5 s)
số n < 200 mà ba cách cho CÙNG một mảng: 199 / 199
12.6 tính chất (đảo, biến đổi tuyến tính, xóa phần tử): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trả về mảng tăng dần (hoặc bất kỳ hoán vị)

Hoán vị `[1, 2, ..., n]` vi phạm ngay: `2 * 2 = 1 + 3`.

In [21]:
n = counters["mảng tăng dần"]
print("n =", n, "| sai ->", wrong_identity(n), "hợp lệ?", is_beautiful(wrong_identity(n), n), "| đúng ->", Solution().beautifulArray(n), is_beautiful(Solution().beautifulArray(n), n))
assert not is_beautiful(wrong_identity(n), n) and is_beautiful(Solution().beautifulArray(n), n)

n = 3 | sai -> [1, 2, 3] hợp lệ? False | đúng -> [1, 3, 2] True


Vì sao sai: các số liên tiếp tạo cấp số cộng. **Đúng:** dựng có cấu trúc (lẻ/chẵn + biến đổi tuyến tính).

### Lỗi 2 — Lẻ rồi chẵn nhưng **không** sắp xếp theo mảng đẹp nhỏ

Đặt số lẻ tăng dần rồi chẵn tăng dần: trong từng nửa các số tạo cấp số cộng.

In [22]:
n = counters["lẻ rồi chẵn tăng dần (không biến đổi)"]
arr = wrong_sorted_odd_even(n)
print("n =", n, "| sai ->", arr, "hợp lệ?", is_beautiful(arr, n))
assert not is_beautiful(arr, n)

n = 5 | sai -> [1, 3, 5, 2, 4] hợp lệ? False


Vì sao sai: bước "ghép lẻ + chẵn" chỉ xử lý cặp lệch; trong từng nửa vẫn phải đẹp, nên nửa lẻ và nửa chẵn phải được sắp theo mảng đẹp nhỏ hơn (`F((n + 1) / 2)`, `F(n / 2)`). **Đúng:** `[2x - 1 for x in F(p)] + [2x for x in F(q)]`.

### Lỗi 3 — Chia sai kích thước hai nửa

Số lẻ trong `1..n` là `(n + 1) // 2`, số chẵn là `n // 2`. Dùng `n // 2` cho cả hai (hoặc ngược) làm mất hoặc thiếu phần tử.

In [23]:
n = counters["chia sai kích thước"]
arr = wrong_split_sizes(n)
print("n =", n, "| sai ->", arr, "| độ dài", len(arr), "| hợp lệ?", is_beautiful(arr, n))
assert not is_beautiful(arr, n)

n = 3 | sai -> [1, 2] | độ dài 2 | hợp lệ? False


Vì sao sai: kết quả không phải hoán vị của `1..n` (thiếu số). **Đúng:** `left = F((n + 1) // 2)`, `right = F(n // 2)`.

### Lỗi 4 — Quên lọc giá trị `> n` khi lặp

Nhân đôi mà không lọc cho ra giá trị vượt `n`.

In [24]:
n = counters["quên lọc giá trị > n"]
arr = wrong_no_filter(n)
print("n =", n, "| sai ->", arr, "| có giá trị > n?", any(x > n for x in arr), "| hợp lệ?", is_beautiful(arr, n))
assert not is_beautiful(arr, n)

n = 5 | sai -> [1, 5, 3, 7, 2] | có giá trị > n? True | hợp lệ? False


Vì sao sai: `2x - 1`, `2x` có thể vượt `n` khi `n` không là lũy thừa của `2`. **Đúng:** lọc `<= n` ở mỗi vòng (hoặc dùng kích thước `(n + 1) // 2`, `n // 2` ở đệ quy).

### Lỗi 5 — So sánh với đáp án mẫu của đề

Đề cho `[2,1,4,3]` ở `n = 4`; lời giải đúng có thể trả `[1,3,2,4]` và vẫn đúng. So bằng `==` với đáp án mẫu là sai cách kiểm thử.

In [25]:
mine = Solution().beautifulArray(4)
print("lời giải:", mine, "| đáp án mẫu: [2, 1, 4, 3] | bằng nhau?", mine == [2, 1, 4, 3], "| cả hai hợp lệ?", is_beautiful(mine, 4), is_beautiful([2, 1, 4, 3], 4))
assert mine != [2, 1, 4, 3] and is_beautiful(mine, 4)

lời giải: [1, 3, 2, 4] | đáp án mẫu: [2, 1, 4, 3] | bằng nhau? False | cả hai hợp lệ? True True


Vì sao: đề nói "trả về **bất kỳ** mảng đẹp". **Đúng:** kiểm tra bằng hàm hợp lệ (`is_beautiful`).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "dựng hoán vị tránh cấp số cộng 3 phần tử" ⇒ **chia để trị theo tính chẵn lẻ**.
- **Hai bổ đề:** (1) `x -> a x + b` giữ tính đẹp; (2) lẻ + chẵn không phạm lẫn nhau vì tổng lẻ.
- Công thức: `F(n) = [2x - 1 for x in F((n + 1) // 2)] + [2x for x in F(n // 2)]`.
- Với bài **nhiều đáp án**: viết **hàm kiểm tra** và kiểm thử `n` từ `1` tới giới hạn.
- Lọc phần tử khỏi một mảng đẹp vẫn đẹp.
- Cùng họ: Construct the Lexicographically Largest Valid Sequence (1718), Permutation Sequence (60), Special Array.
- Test nên có: `n = 1`, các lũy thừa của `2` và lân cận, `n = 1000`.

## 15. Biến thể

1. **Đếm số mảng đẹp** của `n` (đã tính ở mục 5 cho `n <= 8`; dãy này tăng rất nhanh).
2. **Mảng đẹp từ một tập số tùy ý** (không nhất thiết `1..n`): các bổ đề vẫn áp dụng cho dãy số thực khác nhau (bằng ánh xạ tuyến tính).
3. **Mảng đẹp từ mảng nhỏ bằng nhân đôi** (bài tập 5).
4. Bài liên quan: 1718.

Dưới đây là biến thể 2: dựng mảng không có trung bình cộng nằm giữa từ tập số **bất kỳ** (khác nhau) bằng cách ánh xạ.

In [26]:
def beautiful_from_set(values: List[int]) -> List[int]:
    """Sắp xếp tập `values` (đôi một khác nhau) thành dãy không có trung bình cộng nằm giữa: ánh xạ theo chỉ số của mảng đẹp 1..m."""
    ordered = sorted(values)                          # dùng sorted trong biến thể (không thuộc bài nộp)
    m = len(ordered)
    perm = Solution().beautifulArray(m)               # mảng đẹp của 1..m cho thứ hạng
    return [ordered[r - 1] for r in perm]

rng = random.Random(15)
for _ in range(200):
    vals = rng.sample(range(-50, 200), rng.randint(1, 30))
    out = beautiful_from_set(vals)
    assert sorted(out) == sorted(vals)
    # với tập tuỳ ý chỉ có thể bảo đảm đúng khi tập là cấp số cộng; kiểm tra tính chất trên cấp số cộng
for _ in range(100):
    a, d, m = rng.randint(-20, 20), rng.randint(1, 7), rng.randint(1, 40)
    prog = [a + d * t for t in range(m)]
    out = beautiful_from_set(prog)
    assert no_avg_between(out), (a, d, m)
print("cấp số cộng được sắp thành dãy không có trung bình cộng nằm giữa: OK; ví dụ:", beautiful_from_set([10, 20, 30, 40, 50]))

cấp số cộng được sắp thành dãy không có trung bình cộng nằm giữa: OK; ví dụ: [10, 50, 30, 20, 40]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quay lui | Tìm có cắt tỉa | mũ | `O(n)` | Oracle nhỏ |
| 2. Đệ quy chia để trị | `F(n)` từ `F((n+1)/2)`, `F(n/2)` | `O(n)` | `O(n)` | **Cách nộp chính** |
| 3. Lặp nhân đôi | Từ dưới lên, lọc `<= n` | `O(n)` | `O(n)` | Không đệ quy |
| 4. Đảo bit | Đảo bit chỉ số | `O(n log n)` | `O(n)` | Công thức đóng |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
A = [1, 2]
print([2 * x - 1 for x in A], [2 * x for x in A])
print((5 + 1) // 2, 5 // 2)
print(is_beautiful([1, 2, 3], 3), is_beautiful([1, 3, 2], 3))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Dựng bằng tay `F(5)` theo công thức chia để trị: ghi `F(2)`, `F(3)` và `F(5)`.

In [28]:
# Bài 2: điền kết quả tính tay
hand_F2 = None
hand_F3 = None
hand_F5 = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành chia để trị.

In [29]:
def exercise_3_recursive(n: int) -> List[int]:
    if n == 1:
        return [1]
    # TODO 1: left = mảng đẹp của (n + 1) // 2, right = mảng đẹp của n // 2
    # TODO 2: trả về [2x - 1 cho x trong left] + [2x cho x trong right]
    return None


# for n in (1, 2, 5, 16, 100):
#     assert is_beautiful(exercise_3_recursive(n), n)

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_iterative(n)` bằng nhân đôi và lọc `<= n`.

In [30]:
def exercise_4_iterative(n: int) -> List[int]:
    # TODO
    pass


# for n in (1, 2, 5, 16, 100):
#     assert is_beautiful(exercise_4_iterative(n), n)

### Bài 5 — Mini challenge: nhân đôi một mảng đẹp (khó)

Cho `arr` là một mảng đẹp độ dài `m` (hoán vị của `1..m`). Viết `exercise_5_double(arr)` trả về mảng đẹp độ dài `2m` (hoán vị của `1..2m`) **chỉ bằng cách biến đổi `arr`** (không dựng lại từ đầu).

```python
exercise_5_double([1, 2])        -> [1, 3, 2, 4]
exercise_5_double([1, 3, 2])     -> mảng đẹp độ dài 6
```

In [31]:
def exercise_5_double(arr: List[int]) -> List[int]:
    # TODO
    pass


# a = exercise_5_double([1, 2])
# assert is_beautiful(a, 4)
# b = exercise_5_double([1, 3, 2])
# assert is_beautiful(b, 6)

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `2x - 1` cho số lẻ, `2x` cho số chẵn; số lẻ trong `1..n` là `(n + 1) // 2`.
- Mức 2: `[1, 2]` cho lẻ `[1, 3]`, chẵn `[2, 4]`; `n = 5`: lẻ `3`, chẵn `2`.
- Mức 3: `[1, 3] [2, 4]`, `3 2`, `False True`.

**Bài 2**
- Mức 1: `F(1) = [1]`, `F(2) = [1, 2]`.
- Mức 2: `F(3)`: lẻ từ `F(2)` là `[1, 3]`, chẵn từ `F(1)` là `[2]`: `[1, 3, 2]`.
- Mức 3: `F(5)`: lẻ từ `F(3)`: `[1, 5, 3]`; chẵn từ `F(2)`: `[2, 4]`: `[1, 5, 3, 2, 4]`.

**Bài 3**
- Mức 1: `left = exercise_3_recursive((n + 1) // 2)`.
- Mức 2: `right = exercise_3_recursive(n // 2)`.
- Mức 3: `return [2 * x - 1 for x in left] + [2 * x for x in right]`.

**Bài 4**
- Mức 1: bắt đầu `arr = [1]`.
- Mức 2: lặp tới khi `len(arr) == n`.
- Mức 3: `arr = [2*x - 1 for x in arr if 2*x - 1 <= n] + [2*x for x in arr if 2*x <= n]`.

**Bài 5**
- Mức 1: `2m` có `m` số lẻ và `m` số chẵn.
- Mức 2: dùng `arr` cho cả hai nửa.
- Mức 3: `[2x - 1 for x in arr] + [2x for x in arr]`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
A = [1, 2]
print([2 * x - 1 for x in A], [2 * x for x in A])
print((5 + 1) // 2, 5 // 2)
print(is_beautiful([1, 2, 3], 3), is_beautiful([1, 3, 2], 3))

assert ([2 * x - 1 for x in A], [2 * x for x in A]) == ([1, 3], [2, 4])
assert ((5 + 1) // 2, 5 // 2) == (3, 2)
assert (is_beautiful([1, 2, 3], 3), is_beautiful([1, 3, 2], 3)) == (False, True)

[1, 3] [2, 4]
3 2
False True


### Đáp án Bài 2

`F(2) = [1, 2]`, `F(3) = [1, 3, 2]`, `F(5) = [1, 5, 3, 2, 4]`.

In [33]:
hand_F2_solution, hand_F3_solution, hand_F5_solution = [1, 2], [1, 3, 2], [1, 5, 3, 2, 4]
assert beautiful_recursive(2) == hand_F2_solution and beautiful_recursive(3) == hand_F3_solution and beautiful_recursive(5) == hand_F5_solution
assert is_beautiful(hand_F5_solution, 5)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_recursive_solution(n: int) -> List[int]:
    if n == 1:
        return [1]
    left = exercise_3_recursive_solution((n + 1) // 2)
    right = exercise_3_recursive_solution(n // 2)
    return [2 * x - 1 for x in left] + [2 * x for x in right]

for n in (1, 2, 5, 16, 100, 1000):
    assert is_beautiful(exercise_3_recursive_solution(n), n)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_iterative_solution(n: int) -> List[int]:
    arr = [1]
    while len(arr) < n:
        arr = [2 * x - 1 for x in arr if 2 * x - 1 <= n] + [2 * x for x in arr if 2 * x <= n]
    return arr

for n in (1, 2, 5, 16, 100, 1000):
    assert is_beautiful(exercise_4_iterative_solution(n), n)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_double_solution(arr: List[int]) -> List[int]:
    return [2 * x - 1 for x in arr] + [2 * x for x in arr]

assert is_beautiful(exercise_5_double_solution([1, 2]), 4)
assert exercise_5_double_solution([1, 2]) == [1, 3, 2, 4]
assert is_beautiful(exercise_5_double_solution([1, 3, 2]), 6)
for n in range(1, 60):
    a = Solution().beautifulArray(n)
    assert is_beautiful(exercise_5_double_solution(a), 2 * n)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (chia để trị theo tính chẵn lẻ)

```python
class Solution:
    def beautifulArray(self, n: int) -> List[int]:
        if n == 1:
            return [1]
        left = self.beautifulArray((n + 1) // 2)      # (n + 1) // 2 số lẻ
        right = self.beautifulArray(n // 2)           # n // 2 số chẵn
        return [2 * x - 1 for x in left] + [2 * x for x in right]   # lẻ trước, chẵn sau
```

Công thức nhớ nhanh: `F(n) = [2x - 1 | x in F(ceil(n/2))] + [2x | x in F(floor(n/2))]`; `x -> 2x - 1` giữ tính đẹp; lẻ + chẵn không phạm lẫn nhau.

### Checklist trước khi submit

- [ ] Kích thước hai nửa đúng: `(n + 1) // 2` cho số lẻ, `n // 2` cho số chẵn?
- [ ] Hai nửa dựng từ mảng đẹp nhỏ hơn bằng `2x - 1` và `2x` (không phải tăng dần)?
- [ ] Mảng là hoán vị đủ `1..n` (đúng `n` phần tử, không vượt `n`)?
- [ ] Kiểm thử bằng **hàm hợp lệ** (nhiều đáp án), không so đáp án mẫu?
- [ ] Đã thử: `n = 1`, lũy thừa của `2` và lân cận, `n = 1000`?
- [ ] Tên method đúng `beautifulArray(self, n)` và trả `List[int]`?